# 07 — Comprehensive Evaluation & Scientific Ablation Studies (1.3B)
**Goal**: Execute the complete evaluation matrix across **Zero-Shot Base, SFT, PPO-Dense, PPO-Binary, and DPO** for `deepseek-ai/deepseek-coder-1.3b-instruct` and answer all research questions:
- **RQ1**: Can SLMs do agentic debugging ($K=1, 3, 5$) vs. repeated sampling (`F0`)?
- **RQ2**: Does real execution feedback (`F2`/`F3`) outperform no feedback (`F0`), status-only (`F1`), and random fake feedback?
- **RQ3**: Does RL (PPO/DPO) beat SFT and zero-shot baselines in `Pass@1`, `Fix@K`, and `ConditionalRepair@K`?
- **RQ4**: Do SLMs benefit from dense partial-credit rewards (`ppo_dense`) vs. binary pass/fail rewards (`ppo_binary`)?
- **RQ5**: Is DPO (`dpo`) competitive with PPO in multi-turn repair performance and efficiency?
- **RQ6**: At what turn count $K \in \{1, 3, 5, 7\}$ does debugging performance plateau?
- **RQ7**: Does training on APPS generalize out-of-distribution to HumanEval and MBPP?

---

## Step 1: Setup, Universal Path Resolution & Harness Verification

In [1]:
!pip uninstall -y torchao
!pip install -q trl peft accelerate datasets transformers pandas

import sys, os, shutil, importlib, json

def prepare_environment_src():
    if os.path.isdir('/kaggle/working'):
        !rm -rf /kaggle/working/src temp_repo
        !git clone -b Rajdeep https://github.com/Oin19/self-correction-slm.git temp_repo || git clone https://github.com/Oin19/self-correction-slm.git temp_repo || git clone -b junior-A https://github.com/Oin19/self-correction-llm-rl.git temp_repo
        if os.path.exists('temp_repo/src'):
            shutil.copytree('temp_repo/src', '/kaggle/working/src')
            !rm -rf temp_repo
            print("Cloned latest 'src' from Rajdeep branch to /kaggle/working/src")
            return '/kaggle/working'

    curr = os.path.abspath(os.getcwd())
    if os.path.exists(os.path.join(curr, 'src', 'models', 'loader.py')):
        print(f"Using local 'src' directory at {curr}")
        return curr
    
    parent = os.path.abspath('..')
    if os.path.exists(os.path.join(parent, 'src', 'models', 'loader.py')):
        print(f"Using parent 'src' directory at {parent}")
        return parent

    for colab_dir in ['/content/self-correction-slm', '/content/self-correction-llm-rl', '/content']:
        if os.path.exists(os.path.join(colab_dir, 'src', 'models', 'loader.py')):
            print(f"Using Colab directory at {colab_dir}")
            return colab_dir

    if os.path.exists('/kaggle/input'):
        working_src = '/kaggle/working/src'
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'models' in dirs and os.path.exists(os.path.join(root, 'models', 'loader.py')):
                if os.path.exists(working_src):
                    shutil.rmtree(working_src)
                shutil.copytree(root, working_src)
                print(f"Copied 'src' from {root} to {working_src}")
                return '/kaggle/working'
            elif 'src' in dirs and os.path.exists(os.path.join(root, 'src', 'models', 'loader.py')):
                src_dir = os.path.join(root, 'src')
                if os.path.exists(working_src):
                    shutil.rmtree(working_src)
                shutil.copytree(src_dir, working_src)
                print(f"Copied 'src' from {src_dir} to {working_src}")
                return '/kaggle/working'
    return curr

repo_root = prepare_environment_src()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path added: {repo_root}")

try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

import torch
import pandas as pd
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.models.loader import load_model_and_tokenizer
from src.debugging.debug_loop import (
    agentic_debug_loop,
    agentic_loop_no_feedback,
    agentic_loop_repeated_sampling,
)
from src.evaluation.metrics import (
    evaluate,
    calculate_pass_at_1,
    calculate_fix_at_k,
    calculate_error_distribution,
    calculate_recovery_rate,
    summarize_eval_metrics,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("./results", exist_ok=True)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# Auto-recover any previously saved JSONL traces from /kaggle/input for instant resume
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for fn in files:
            if fn.endswith('.jsonl') and ('humaneval' in fn or 'mbpp' in fn or 'ablation_' in fn):
                dst = os.path.join('./results', fn)
                if not os.path.exists(dst):
                    shutil.copy2(os.path.join(root, fn), dst)
                    print(f"Recovered cached trace for resume: {dst}")

print("Loading evaluation benchmarks (HumanEval & MBPP)...")
humaneval = load_dataset('openai/openai_humaneval', split='test')
mbpp = load_dataset('google-research-datasets/mbpp', split='test')

print(f"Loaded HumanEval: {len(humaneval)} test cases | MBPP: {len(mbpp)} test cases.")


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.4 MB/s eta 0:00:0000:010:01
Cloning into 'temp_repo'...
remote: Enumerating objects: 392, done.
remote: Counting objects: 100% (392/392), done.
remote: Compressing objects: 100% (228/228), done.
remote: Total 392 (delta 190), reused 336 (delta 134), pack-reused 0 (from 0)
Receiving objects: 100% (392/392), 2.39 MiB | 47.01 MiB/s, done.
Resolving deltas: 100% (190/190), done.
Cloned latest 'src' from Rajdeep branch to /kaggle/working/src
Project path added: /kaggle/working
Recovered cached trace for resume: ./results/dpo_humaneval_k5.jsonl
Recovered cached trace for resume: ./results/dpo_mbpp_k5.jsonl


Loading evaluation benchmarks (HumanEval & MBPP)...


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

... (5 outputs left out)

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/10 [00:00<?, ? examples/s]

Loaded HumanEval: 164 test cases | MBPP: 500 test cases.


## Step 2: Main Evaluation Grid — Zero-Shot vs SFT vs PPO-Dense vs PPO-Binary vs DPO (RQ3, RQ4, RQ5, RQ7)
Computes `Pass@1`, `Fix@1`, `Fix@3`, `Fix@5`, `ConditionalRepair@3`, and `ConditionalRepair@5` on HumanEval and MBPP across all 5 model arms (`Zero-Shot Base`, `SFT Baseline`, `PPO-Dense`, `PPO-Binary`, `DPO Model`) and saves per-problem JSONL traces plus `./results/final_evaluation_grid.csv`.

In [2]:
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
EVAL_SLICE = None  # Full benchmark evaluation: HumanEval (164) & MBPP (500)
MAX_K = 5
MAX_NEW_TOKENS = 320  # Early-stops automatically at closing ``` code fence

def get_optimal_batch_size():
    if not torch.cuda.is_available():
        return 1
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    gpu_name = torch.cuda.get_device_name(0)
    bs = 32 if vram_gb >= 40 else (16 if vram_gb >= 14 else 8)
    print(f"Detected GPU 0: {gpu_name} ({vram_gb:.1f} GB VRAM) -> Using BATCH_SIZE={bs}")
    return bs

BATCH_SIZE = get_optimal_batch_size()

he_eval_ds = humaneval.select(range(min(EVAL_SLICE, len(humaneval)))) if EVAL_SLICE else humaneval
mbpp_eval_ds = mbpp.select(range(min(EVAL_SLICE, len(mbpp)))) if EVAL_SLICE else mbpp

def find_checkpoint(name_tags):
    if isinstance(name_tags, str):
        name_tags = [name_tags]
    candidates = []
    for name_tag in name_tags:
        tag_l = name_tag.lower()
        direct_paths = [
            f"./checkpoints/{name_tag}/final",
            f"/kaggle/working/checkpoints/{name_tag}/final",
            f"/content/checkpoints/{name_tag}/final",
            f"/content/self-correction-slm/checkpoints/{name_tag}/final",
            f"/content/self-correction-llm-rl/checkpoints/{name_tag}/final",
            f"/content/drive/MyDrive/checkpoints/{name_tag}/final",
        ]
        for p in direct_paths:
            if os.path.exists(p) and (
                os.path.exists(os.path.join(p, 'adapter_config.json'))
                or os.path.exists(os.path.join(p, 'config.json'))
            ):
                return p
        for search_root in ['./checkpoints', '/kaggle/working/checkpoints', '/kaggle/input']:
            if not os.path.exists(search_root):
                continue
            for root, dirs, files in os.walk(search_root):
                if 'adapter_config.json' in files or 'config.json' in files:
                    rel_parts = [pt.lower() for pt in os.path.normpath(root).split(os.sep)[-3:]]
                    if any(pt == tag_l or pt.startswith(tag_l) for pt in rel_parts):
                        candidates.append(root)
        if candidates:
            break
    if not candidates:
        return None
    final_cands = [c for c in candidates if os.path.basename(c).lower() == 'final']
    return final_cands[0] if final_cands else sorted(candidates)[-1]

def load_eval_model(checkpoint_path, gpu_id=0):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    # Pin the 1.3B model (~2.6 GB FP16) onto a single GPU so multi-GPU never incurs cross-GPU PCIe latency
    dev_map = {"": gpu_id} if torch.cuda.is_available() else None
    dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (
        torch.float16 if torch.cuda.is_available() else torch.float32
    )
    if checkpoint_path and os.path.exists(os.path.join(checkpoint_path, 'adapter_config.json')):
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=dtype,
            device_map=dev_map,
            trust_remote_code=True,
        )
        model = PeftModel.from_pretrained(base_model, checkpoint_path, is_trainable=False)
        model = model.merge_and_unload()  # Fuse LoRA weights into base layers for ~25% faster decoding
        print(f"Loaded & merged trained LoRA adapter from: {checkpoint_path}")
    else:
        target = checkpoint_path if (checkpoint_path and os.path.exists(checkpoint_path)) else MODEL_NAME
        model = AutoModelForCausalLM.from_pretrained(
            target,
            torch_dtype=dtype,
            device_map=dev_map,
            trust_remote_code=True,
        )
        print(f"Loaded base/full model from: {target}")
    model.eval()
    return model, tokenizer

checkpoints_to_eval = [
    ("Zero-Shot Base", MODEL_NAME),
    ("SFT Baseline", find_checkpoint("sft")),
    ("PPO-Dense Model", find_checkpoint(["ppo_dense", "ppo"])),
    ("PPO-Binary Model", find_checkpoint("ppo_binary")),
    ("DPO Model", find_checkpoint("dpo")),
]

from concurrent.futures import ThreadPoolExecutor
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"Available GPUs for data-parallel evaluation: {NUM_GPUS}")

results_grid = []
summary_by_model = {}

for label, path in checkpoints_to_eval:
    if label != "Zero-Shot Base" and not path:
        print(f"\n[WARNING] Checkpoint for '{label}' not found in local/Kaggle paths — skipping so we do not falsely report Base Model numbers as '{label}'.")
        continue
    print(f"\n================ Evaluating: {label} ({path}) ================")
    slug = label.lower().replace(" ", "_").replace("-", "_")

    if NUM_GPUS >= 2:
        # 2-GPU Data Parallelism: run HumanEval on GPU 0 and MBPP on GPU 1 simultaneously
        def _run_on_gpu(gpu_id, ds, bench_tag, out_path):
            m, tok = load_eval_model(path, gpu_id=gpu_id)
            res = evaluate(
                m, tok, ds, K=MAX_K,
                label=f"{label} {bench_tag} [GPU{gpu_id}]",
                output_jsonl=out_path,
                batch_size=BATCH_SIZE,
                max_new_tokens=MAX_NEW_TOKENS,
                resume=True,
            )
            del m
            torch.cuda.empty_cache()
            return res

        with ThreadPoolExecutor(max_workers=2) as pool:
            fut_he = pool.submit(_run_on_gpu, 0, he_eval_ds, "HE", f"./results/{slug}_humaneval_k{MAX_K}.jsonl")
            fut_mbpp = pool.submit(_run_on_gpu, 1, mbpp_eval_ds, "MBPP", f"./results/{slug}_mbpp_k{MAX_K}.jsonl")
            he_res = fut_he.result()
            mbpp_res = fut_mbpp.result()
    else:
        model, tokenizer = load_eval_model(path, gpu_id=0)
        he_res = evaluate(
            model, tokenizer, he_eval_ds, K=MAX_K,
            label=f"{label} HE",
            output_jsonl=f"./results/{slug}_humaneval_k{MAX_K}.jsonl",
            batch_size=BATCH_SIZE,
            max_new_tokens=MAX_NEW_TOKENS,
            resume=True,
        )
        mbpp_res = evaluate(
            model, tokenizer, mbpp_eval_ds, K=MAX_K,
            label=f"{label} MBPP",
            output_jsonl=f"./results/{slug}_mbpp_k{MAX_K}.jsonl",
            batch_size=BATCH_SIZE,
            max_new_tokens=MAX_NEW_TOKENS,
            resume=True,
        )
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    summary_by_model[label] = {"he": he_res, "mbpp": mbpp_res, "path": path}

    for k in [1, 3, 5]:
        results_grid.append({
            "Model": label,
            "Checkpoint": path,
            "K": k,
            "HumanEval Pass@1": f"{he_res['pass_at_1']:.2%}",
            f"HumanEval Fix@K": f"{he_res.get(f'fix_at_{k}', 0.0):.2%}",
            f"HumanEval CondRepair@K": f"{he_res.get(f'conditional_repair_at_{k}', 0.0):.2%}",
            "MBPP Pass@1": f"{mbpp_res['pass_at_1']:.2%}",
            f"MBPP Fix@K": f"{mbpp_res.get(f'fix_at_{k}', 0.0):.2%}",
            f"MBPP CondRepair@K": f"{mbpp_res.get(f'conditional_repair_at_{k}', 0.0):.2%}",
        })

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    pd.DataFrame(results_grid).to_csv("./results/final_evaluation_grid.csv", index=False)
    print(f"[Saved incremental progress for {label} to ./results/final_evaluation_grid.csv]")

df_results = pd.DataFrame(results_grid)
df_results.to_csv("./results/final_evaluation_grid.csv", index=False)
print("\n================ AAAI Final Evaluation Grid Results ================")
print(df_results.to_string(index=False))
print("\nSaved full evaluation table to ./results/final_evaluation_grid.csv")


Detected GPU 0: Tesla T4 (14.6 GB VRAM) -> Using BATCH_SIZE=16
Available GPUs for data-parallel evaluation: 2

================ Evaluating: Zero-Shot Base (deepseek-ai/deepseek-coder-1.3b-instruct) ================


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

... (30 outputs left out)

<unknown>:3: SyntaxWarning: invalid escape sequence '\d'
<unknown>:7: SyntaxWarning: invalid escape sequence '\w'


DPO Model HE [GPU0] [144/164] Pass@1=63.19% Fix@5=69.44% CondRepair@5=16.98% (229.8s/16 probs)
DPO Model HE [GPU0] [160/164] Pass@1=63.75% Fix@5=69.38% CondRepair@5=15.52% (122.4s/16 probs)
DPO Model MBPP [GPU1] [176/500] Pass@1=36.36% Fix@5=43.75% CondRepair@5=11.61% (154.2s/16 probs)
DPO Model HE [GPU0] [164/164] Pass@1=63.41% Fix@5=68.90% CondRepair@5=15.00% (81.1s/4 probs)
DPO Model MBPP [GPU1] [192/500] Pass@1=36.98% Fix@5=43.75% CondRepair@5=10.74% (197.5s/16 probs)
DPO Model MBPP [GPU1] [208/500] Pass@1=36.06% Fix@5=42.79% CondRepair@5=10.53% (139.4s/16 probs)
DPO Model MBPP [GPU1] [224/500] Pass@1=37.50% Fix@5=44.20% CondRepair@5=10.71% (93.8s/16 probs)
DPO Model MBPP [GPU1] [240/500] Pass@1=37.08% Fix@5=43.75% CondRepair@5=10.60% (148.3s/16 probs)
DPO Model MBPP [GPU1] [256/500] Pass@1=36.72% Fix@5=43.36% CondRepair@5=10.49% (140.4s/16 probs)
DPO Model MBPP [GPU1] [272/500] Pass@1=37.13% Fix@5=43.75% CondRepair@5=10.53% (95.1s/16 probs)
DPO Model MBPP [GPU1] [288/500] Pass@1=3

<unknown>:6: SyntaxWarning: invalid escape sequence '\s'
<unknown>:6: SyntaxWarning: invalid escape sequence '\s'


DPO Model MBPP [GPU1] [416/500] Pass@1=36.78% Fix@5=42.07% CondRepair@5=8.37% (149.9s/16 probs)
DPO Model MBPP [GPU1] [432/500] Pass@1=36.81% Fix@5=41.90% CondRepair@5=8.06% (98.9s/16 probs)
DPO Model MBPP [GPU1] [448/500] Pass@1=36.83% Fix@5=41.96% CondRepair@5=8.13% (93.3s/16 probs)
DPO Model MBPP [GPU1] [464/500] Pass@1=37.50% Fix@5=42.67% CondRepair@5=8.28% (119.5s/16 probs)
DPO Model MBPP [GPU1] [480/500] Pass@1=38.12% Fix@5=43.12% CondRepair@5=8.08% (220.8s/16 probs)
DPO Model MBPP [GPU1] [496/500] Pass@1=39.11% Fix@5=43.95% CondRepair@5=7.95% (87.3s/16 probs)
DPO Model MBPP [GPU1] [500/500] Pass@1=39.20% Fix@5=44.00% CondRepair@5=7.89% (101.1s/4 probs)
[Saved incremental progress for DPO Model to ./results/final_evaluation_grid.csv]

================ AAAI Final Evaluation Grid Results ================
           Model                                                                    Checkpoint  K HumanEval Pass@1 HumanEval Fix@K HumanEval CondRepair@K MBPP Pass@1 MBPP Fix@K MBP

## Step 3: Scientific Ablations (RQ1/RQ2 Feedback Controls, RQ4 Dense vs Binary, RQ6 K-Turn Plateau)
1. **RQ1 & RQ2 (Feedback Utility & Repeated-Sampling Control)**: Real execution feedback (`F2`) vs. No-feedback repeated sampling (`F0` / `B1`) vs. Status-only (`F1`) vs. Structured (`F3`) vs. Random fake error (`agentic_loop_no_feedback`)
2. **RQ4 (Reward Granularity)**: Dense partial-test reward (`PPO-Dense`) vs. Binary pass/fail reward (`PPO-Binary`)
3. **RQ6 (Turn Count Plateau)**: `Fix@K` and `ConditionalRepair@K` trajectory across $K \in \{1, 3, 5, 7\}$

In [3]:
# Choose the best available trained checkpoint (DPO / PPO-Dense / SFT / Base) for ablations
ablation_ckpt = (
    find_checkpoint("dpo")
    or find_checkpoint(["ppo_dense", "ppo"])
    or find_checkpoint("sft")
    or MODEL_NAME
)
print(f"\n--- Running Ablations using checkpoint: {ablation_ckpt} ---")
ABLATION_SLICE = None  # Full 164 HumanEval test set for ablations
ablation_ds = humaneval.select(range(min(ABLATION_SLICE, len(humaneval)))) if ABLATION_SLICE else humaneval

print("\n--- 1. RQ1, RQ2 & RQ6 Batched Feedback & K=7 Trajectory Ablations ---")
# Run K=7 under F2 once; its first 3 turns also supply exact F2 (K=3) metrics with zero duplicate compute
if NUM_GPUS >= 2:
    # Split ablations across GPU 0 and GPU 1 in parallel
    def _run_abl_gpu0():
        m0, tok0 = load_eval_model(ablation_ckpt, gpu_id=0)
        r_k7 = evaluate(
            m0, tok0, ablation_ds, K=7, label="F2 Sanitized Traceback Feedback [GPU0]",
            feedback_mode="F2", batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
            output_jsonl="./results/ablation_k7_trajectory.jsonl", resume=True,
        )
        r_f1 = evaluate(
            m0, tok0, ablation_ds, K=3, label="F1 Status-Only Feedback [GPU0]",
            feedback_mode="F1", batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
            output_jsonl="./results/ablation_f1_status_only.jsonl", resume=True,
        )
        del m0; torch.cuda.empty_cache()
        return r_k7, r_f1

    def _run_abl_gpu1():
        m1, tok1 = load_eval_model(ablation_ckpt, gpu_id=1)
        r_f0 = evaluate(
            m1, tok1, ablation_ds, K=3, label="F0 Repeated Sampling (No Feedback) [GPU1]",
            debug_loop_fn=agentic_loop_repeated_sampling, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
            output_jsonl="./results/ablation_f0_repeated_sampling.jsonl", resume=True,
        )
        r_f3 = evaluate(
            m1, tok1, ablation_ds, K=3, label="F3 Structured Feedback [GPU1]",
            feedback_mode="F3", batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
            output_jsonl="./results/ablation_f3_structured.jsonl", resume=True,
        )
        r_fake = evaluate(
            m1, tok1, ablation_ds, K=3, label="Random Fake Error Feedback [GPU1]",
            debug_loop_fn=agentic_loop_no_feedback, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
            output_jsonl="./results/ablation_fake_feedback.jsonl", resume=True,
        )
        del m1; torch.cuda.empty_cache()
        return r_f0, r_f3, r_fake

    with ThreadPoolExecutor(max_workers=2) as pool:
        f0_fut = pool.submit(_run_abl_gpu0)
        f1_fut = pool.submit(_run_abl_gpu1)
        res_k7, res_f1 = f0_fut.result()
        res_f0, res_f3, res_fake = f1_fut.result()
    res_f2 = res_k7
else:
    model, tokenizer = load_eval_model(ablation_ckpt, gpu_id=0)
    res_k7 = evaluate(
        model, tokenizer, ablation_ds, K=7, label="F2 Sanitized Traceback Feedback",
        feedback_mode="F2",
        batch_size=BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        output_jsonl="./results/ablation_k7_trajectory.jsonl",
        resume=True,
    )
    res_f2 = res_k7
    res_f0 = evaluate(
        model, tokenizer, ablation_ds, K=3, label="F0 Repeated Sampling (No Feedback)",
        debug_loop_fn=agentic_loop_repeated_sampling,
        batch_size=BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        output_jsonl="./results/ablation_f0_repeated_sampling.jsonl",
        resume=True,
    )
    res_f1 = evaluate(
        model, tokenizer, ablation_ds, K=3, label="F1 Status-Only Feedback",
        feedback_mode="F1",
        batch_size=BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        output_jsonl="./results/ablation_f1_status_only.jsonl",
        resume=True,
    )
    res_f3 = evaluate(
        model, tokenizer, ablation_ds, K=3, label="F3 Structured Feedback",
        feedback_mode="F3",
        batch_size=BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        output_jsonl="./results/ablation_f3_structured.jsonl",
        resume=True,
    )
    res_fake = evaluate(
        model, tokenizer, ablation_ds, K=3, label="Random Fake Error Feedback",
        debug_loop_fn=agentic_loop_no_feedback,
        batch_size=BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        output_jsonl="./results/ablation_fake_feedback.jsonl",
        resume=True,
    )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Materialize explicit K=3 F2 JSONL file from the shared K=7 F2 rollout so all 5 ablation JSONLs exist separately
with open("./results/ablation_f2_traceback.jsonl", "w", encoding="utf-8") as _f2_out:
    for _r in res_k7.get("records", []):
        _st3 = _r.get("statuses", [])[:3]
        _ac3 = "AC" in _st3
        _f2_rec = {
            **_r,
            "label": "F2 Sanitized Traceback Feedback",
            "K": 3,
            "solved": _ac3,
            "solved_turn": (_st3.index("AC") + 1) if _ac3 else None,
            "statuses": _st3,
            "final_status": _st3[-1] if _st3 else "CE",
            "history": _r.get("history", [])[:3],
        }
        _f2_out.write(json.dumps(_f2_rec) + "\n")

df_feedback = pd.DataFrame([
    {"Condition": r["label"], "Pass@1": f"{r['pass_at_1']:.2%}", "Fix@3": f"{r['fix_at_3']:.2%}", "CondRepair@3": f"{r['conditional_repair_at_3']:.2%}"}
    for r in [res_f0, res_f1, res_f2, res_f3, res_fake]
])
df_feedback.to_csv("./results/ablation_feedback_modes.csv", index=False)
print("\n=== Feedback Granularity Ablation Results (RQ1 & RQ2) ===")
print(df_feedback.to_string(index=False))

k_rows = []
for k_val in [1, 3, 5, 7]:
    recs = res_k7["records"]
    total_n = len(recs)
    init_fails = [r for r in recs if not r["pass_at_1"]]
    fix_k = sum(1 for r in recs if "AC" in r["statuses"][:k_val]) / total_n if total_n else 0.0
    cond_k = sum(1 for r in init_fails if "AC" in r["statuses"][:k_val]) / len(init_fails) if init_fails else 0.0
    k_rows.append({"K": k_val, "Fix@K": f"{fix_k:.2%}", "ConditionalRepair@K": f"{cond_k:.2%}"})

df_k = pd.DataFrame(k_rows)
df_k.to_csv("./results/ablation_k_plateau.csv", index=False)
print("\n=== RQ6 K-Turn Trajectory Results ===")
print(df_k.to_string(index=False))

print("\nAll evaluation and ablation studies finished successfully! Artifacts saved in ./results/")



--- Running Ablations using checkpoint: /kaggle/input/notebooks/rjd2002/notebook6-dpo/checkpoints/dpo/final ---

--- 1. RQ1, RQ2 & RQ6 Batched Feedback & K=7 Trajectory Ablations ---


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loaded & merged trained LoRA adapter from: /kaggle/input/notebooks/rjd2002/notebook6-dpo/checkpoints/dpo/final
Loaded & merged trained LoRA adapter from: /kaggle/input/notebooks/rjd2002/notebook6-dpo/checkpoints/dpo/final
F2 Sanitized Traceback Feedback [GPU0] [16/164] Pass@1=93.75% Fix@7=100.00% CondRepair@7=100.00% (43.1s/16 probs)
F0 Repeated Sampling (No Feedback) [GPU1] [16/164] Pass@1=93.75% Fix@3=93.75% CondRepair@3=0.00% (51.7s/16 probs)
F0 Repeated Sampling (No Feedback) [GPU1] [32/164] Pass@1=93.75% Fix@3=93.75% CondRepair@3=0.00% (30.3s/16 probs)
F2 Sanitized Traceback Feedback [GPU0] [32/164] Pass@1=93.75% Fix@7=96.88% CondRepair@7=50.00% (54.0s/16 probs)
F0 Repeated Sampling (No Feedback) [GPU1] [48/164] Pass@1=89.58% Fix@3=91.67% CondRepair@3=20.00% (111.8s/16 probs)
F0 Repeated Sampling (No Feedback) [GPU1] [64/164] Pass@1=89.06% Fix@3=92.19% CondRepair@3=28.57% (30.6s/16 probs)
F2 Sanitized Traceback Feedback [GPU0] [48/164] Pass@1=89.58% Fix@7=93.75% CondRepair@7=40.00

<unknown>:3: SyntaxWarning: invalid escape sequence '\d'
<unknown>:4: SyntaxWarning: invalid escape sequence '\d'


F0 Repeated Sampling (No Feedback) [GPU1] [144/164] Pass@1=63.19% Fix@3=73.61% CondRepair@3=28.30% (138.3s/16 probs)
F0 Repeated Sampling (No Feedback) [GPU1] [160/164] Pass@1=63.75% Fix@3=73.75% CondRepair@3=27.59% (85.0s/16 probs)
F2 Sanitized Traceback Feedback [GPU0] [96/164] Pass@1=75.00% Fix@7=81.25% CondRepair@7=25.00% (320.4s/16 probs)
F0 Repeated Sampling (No Feedback) [GPU1] [164/164] Pass@1=63.41% Fix@3=73.17% CondRepair@3=26.67% (62.1s/4 probs)
F3 Structured Feedback [GPU1] [16/164] Pass@1=93.75% Fix@3=100.00% CondRepair@3=100.00% (40.5s/16 probs)
F3 Structured Feedback [GPU1] [32/164] Pass@1=93.75% Fix@3=96.88% CondRepair@3=50.00% (33.3s/16 probs)
F3 Structured Feedback [GPU1] [48/164] Pass@1=89.58% Fix@3=91.67% CondRepair@3=20.00% (117.1s/16 probs)
F2 Sanitized Traceback Feedback [GPU0] [112/164] Pass@1=70.54% Fix@7=76.79% CondRepair@7=21.21% (276.6s/16 probs)
F3 Structured Feedback [GPU1] [64/164] Pass@1=89.06% Fix@3=90.62% CondRepair@3=14.29% (54.4s/16 probs)
F3 Structu

... (8 outputs left out)

<unknown>:3: SyntaxWarning: invalid escape sequence '\d'
<unknown>:3: SyntaxWarning: invalid escape sequence '\d'


Random Fake Error Feedback [GPU1] [164/164] Pass@1=63.41% Fix@3=67.68% CondRepair@3=11.67% (59.0s/4 probs)


<unknown>:3: SyntaxWarning: invalid escape sequence '\d'


F1 Status-Only Feedback [GPU0] [144/164] Pass@1=63.19% Fix@3=66.67% CondRepair@3=9.43% (138.1s/16 probs)
F1 Status-Only Feedback [GPU0] [160/164] Pass@1=63.75% Fix@3=68.12% CondRepair@3=12.07% (93.8s/16 probs)
F1 Status-Only Feedback [GPU0] [164/164] Pass@1=63.41% Fix@3=67.68% CondRepair@3=11.67% (52.3s/4 probs)

=== Feedback Granularity Ablation Results (RQ1 & RQ2) ===
                                Condition Pass@1  Fix@3 CondRepair@3
F0 Repeated Sampling (No Feedback) [GPU1] 63.41% 73.17%       26.67%
           F1 Status-Only Feedback [GPU0] 63.41% 67.68%       11.67%
   F2 Sanitized Traceback Feedback [GPU0] 63.41% 65.85%        6.67%
            F3 Structured Feedback [GPU1] 63.41% 65.24%        5.00%
        Random Fake Error Feedback [GPU1] 63.41% 67.68%       11.67%

=== RQ6 K-Turn Trajectory Results ===
 K  Fix@K ConditionalRepair@K
 1 63.41%               0.00%
 3 65.85%               6.67%
 5 67.07%              10.00%
 7 68.29%              13.33%

All evaluation and abla

## Step 4: Final Unified Master Table — All Models, Parameters & Benchmark Metrics
Consolidates model architecture parameters (`Total Params`, `Trainable LoRA Params`, `LoRA Config`), training hyperparameters (`LR`, `Epochs/Steps`, `Reward/Beta`, `KL`), evaluation decoding parameters, and full HumanEval + MBPP metrics (`Pass@1`, `Fix@1`, `Fix@3`, `Fix@5`, `CondRepair@3`, `CondRepair@5`) into publication-ready tables.

In [4]:
import os, json
import pandas as pd

BASE_TOTAL_PARAMS = 1_346_471_936  # deepseek-ai/deepseek-coder-1.3b-instruct exact parameter count

def _count_safetensors_params(ckpt_dir):
    if not ckpt_dir or not os.path.isdir(ckpt_dir):
        return 0
    st_path = os.path.join(ckpt_dir, "adapter_model.safetensors")
    if os.path.isfile(st_path):
        try:
            from safetensors import safe_open
            total = 0
            with safe_open(st_path, framework="pt", device="cpu") as f:
                for k in f.keys():
                    shape = f.get_slice(k).get_shape()
                    n = 1
                    for dim in shape:
                        n *= dim
                    total += n
            return total
        except Exception:
            pass
    return 0

def _extract_model_params_info(label, ckpt_path):
    if label == "Zero-Shot Base" or not ckpt_path or ckpt_path == MODEL_NAME:
        return {
            "Base Architecture": MODEL_NAME,
            "Total Params": f"{BASE_TOTAL_PARAMS / 1e9:.3f}B ({BASE_TOTAL_PARAMS:,})",
            "Trainable Params": "0 (0.00%)",
            "LoRA Config": "None (Frozen Base)",
            "Training Hyperparameters": "Zero-shot pretrained (no fine-tuning)",
            "Eval Decoding": f"T1: Greedy | T2..{MAX_K}: temp=1.0, top_p=0.95, max_tok={MAX_NEW_TOKENS}",
        }

    adapter_cfg = {}
    ac_path = os.path.join(ckpt_path, "adapter_config.json")
    if os.path.isfile(ac_path):
        with open(ac_path, encoding="utf-8") as f:
            adapter_cfg = json.load(f)

    ppo_meta = {}
    pm_path = os.path.join(ckpt_path, "ppo_metadata.json")
    if os.path.isfile(pm_path):
        with open(pm_path, encoding="utf-8") as f:
            ppo_meta = json.load(f)

    dpo_meta = {}
    dm_path = os.path.join(ckpt_path, "dpo_metadata.json")
    if os.path.isfile(dm_path):
        with open(dm_path, encoding="utf-8") as f:
            dpo_meta = json.load(f)

    trainable_n = ppo_meta.get("trainable_parameters") or _count_safetensors_params(ckpt_path) or 3_145_728
    total_n = ppo_meta.get("total_parameters") or (BASE_TOTAL_PARAMS + trainable_n)
    pct = 100.0 * trainable_n / total_n

    r_val = adapter_cfg.get("r", 16)
    alpha_val = adapter_cfg.get("lora_alpha", 32)
    drop_val = adapter_cfg.get("lora_dropout", 0.05)
    mods = ",".join(sorted(adapter_cfg.get("target_modules", ["q_proj", "v_proj"])))
    lora_str = f"r={r_val}, alpha={alpha_val}, drop={drop_val} ({mods})"

    if "SFT" in label:
        hp_str = "LR=2e-5, epochs=1, batch=2x4, seed=42 (APPS 2k)"
    elif "PPO" in label:
        rmode = ppo_meta.get("reward_mode", "dense" if "Dense" in label else "binary")
        lr = ppo_meta.get("learning_rate", 2e-7)
        steps = ppo_meta.get("max_steps", 100)
        kl = ppo_meta.get("init_kl_coef", 0.30)
        tkl = ppo_meta.get("target_kl", 4.0)
        seed = ppo_meta.get("seed", 42)
        hp_str = f"reward={rmode}, LR={lr}, steps={steps}, init_KL={kl}, target_KL={tkl}, seed={seed}"
    elif "DPO" in label:
        beta = dpo_meta.get("beta", 0.1)
        lr = dpo_meta.get("learning_rate", 5e-5)
        ep = dpo_meta.get("num_train_epochs", 3)
        bs = dpo_meta.get("per_device_train_batch_size", 4)
        npairs = dpo_meta.get("num_preference_pairs", "N/A")
        seed = dpo_meta.get("seed", 42)
        hp_str = f"beta={beta}, LR={lr}, epochs={ep}, batch={bs}, pairs={npairs}, seed={seed}"
    else:
        hp_str = "Custom checkpoint"

    return {
        "Base Architecture": MODEL_NAME,
        "Total Params": f"{total_n / 1e9:.3f}B ({total_n:,})",
        "Trainable Params": f"{trainable_n / 1e6:.2f}M ({pct:.2f}%)",
        "LoRA Config": lora_str,
        "Training Hyperparameters": hp_str,
        "Eval Decoding": f"T1: Greedy | T2..{MAX_K}: temp=1.0, top_p=0.95, max_tok={MAX_NEW_TOKENS}",
    }

def _metrics_from_jsonl(jsonl_path):
    if not os.path.isfile(jsonl_path):
        return None
    recs = []
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                recs.append(json.loads(line))
    if not recs:
        return None
    n = len(recs)
    p1 = sum(1 for r in recs if r.get("pass_at_1")) / n
    f1 = sum(1 for r in recs if "AC" in r.get("statuses", [])[:1]) / n
    f3 = sum(1 for r in recs if "AC" in r.get("statuses", [])[:3]) / n
    f5 = sum(1 for r in recs if "AC" in r.get("statuses", [])[:5]) / n
    fails = [r for r in recs if not r.get("pass_at_1")]
    nf = len(fails)
    cr1 = 0.0
    cr3 = (sum(1 for r in fails if "AC" in r.get("statuses", [])[:3]) / nf) if nf else 0.0
    cr5 = (sum(1 for r in fails if "AC" in r.get("statuses", [])[:5]) / nf) if nf else 0.0
    return {
        "pass_at_1": p1, "fix_at_1": f1, "fix_at_3": f3, "fix_at_5": f5,
        "conditional_repair_at_1": cr1, "conditional_repair_at_3": cr3, "conditional_repair_at_5": cr5,
        "total": n,
    }

master_rows = []
detailed_k_rows = []

for label, path in checkpoints_to_eval:
    slug = label.lower().replace(" ", "_").replace("-", "_")
    if 'summary_by_model' in dir() and label in summary_by_model:
        he_m = summary_by_model[label]["he"]
        mbpp_m = summary_by_model[label]["mbpp"]
    else:
        he_m = _metrics_from_jsonl(f"./results/{slug}_humaneval_k{MAX_K}.jsonl")
        mbpp_m = _metrics_from_jsonl(f"./results/{slug}_mbpp_k{MAX_K}.jsonl")
    if not he_m or not mbpp_m:
        continue

    pinfo = _extract_model_params_info(label, path)
    master_rows.append({
        "Model": label,
        "Total Params": pinfo["Total Params"],
        "Trainable Params": pinfo["Trainable Params"],
        "LoRA Config": pinfo["LoRA Config"],
        "Training Hyperparameters": pinfo["Training Hyperparameters"],
        "Eval Decoding": pinfo["Eval Decoding"],
        "HE Pass@1": f"{he_m['pass_at_1']:.2%}",
        "HE Fix@3": f"{he_m.get('fix_at_3', 0.0):.2%}",
        "HE Fix@5": f"{he_m.get('fix_at_5', 0.0):.2%}",
        "HE CondRepair@3": f"{he_m.get('conditional_repair_at_3', 0.0):.2%}",
        "HE CondRepair@5": f"{he_m.get('conditional_repair_at_5', 0.0):.2%}",
        "MBPP Pass@1": f"{mbpp_m['pass_at_1']:.2%}",
        "MBPP Fix@3": f"{mbpp_m.get('fix_at_3', 0.0):.2%}",
        "MBPP Fix@5": f"{mbpp_m.get('fix_at_5', 0.0):.2%}",
        "MBPP CondRepair@3": f"{mbpp_m.get('conditional_repair_at_3', 0.0):.2%}",
        "MBPP CondRepair@5": f"{mbpp_m.get('conditional_repair_at_5', 0.0):.2%}",
    })

    for k in [1, 3, 5]:
        detailed_k_rows.append({
            "Model": label,
            "K": k,
            "Total Params": pinfo["Total Params"],
            "Trainable Params": pinfo["Trainable Params"],
            "LoRA Config": pinfo["LoRA Config"],
            "Training Hyperparameters": pinfo["Training Hyperparameters"],
            "HumanEval Pass@1": f"{he_m['pass_at_1']:.2%}",
            "HumanEval Fix@K": f"{he_m.get(f'fix_at_{k}', 0.0):.2%}",
            "HumanEval CondRepair@K": f"{he_m.get(f'conditional_repair_at_{k}', 0.0):.2%}",
            "MBPP Pass@1": f"{mbpp_m['pass_at_1']:.2%}",
            "MBPP Fix@K": f"{mbpp_m.get(f'fix_at_{k}', 0.0):.2%}",
            "MBPP CondRepair@K": f"{mbpp_m.get(f'conditional_repair_at_{k}', 0.0):.2%}",
        })

df_master = pd.DataFrame(master_rows)
df_detailed_k = pd.DataFrame(detailed_k_rows)

df_master.to_csv("./results/final_models_and_parameters_table.csv", index=False)
df_detailed_k.to_csv("./results/final_evaluation_grid_with_params.csv", index=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)

print("\n" + "=" * 110)
print("FINAL MASTER TABLE: ALL MODELS, PARAMETERS & BENCHMARK RESULTS (HumanEval N=164 | MBPP N=500)")
print("=" * 110)
print(df_master.to_string(index=False))

print("\n" + "=" * 110)
print("DETAILED K-TURN BREAKDOWN WITH MODEL PARAMETERS (K = 1, 3, 5)")
print("=" * 110)
print(df_detailed_k.to_string(index=False))

try:
    from IPython.display import display
    display(df_master)
except Exception:
    pass

print("\nSaved CSV tables:")
print("  1. ./results/final_models_and_parameters_table.csv")
print("  2. ./results/final_evaluation_grid_with_params.csv")



FINAL MASTER TABLE: ALL MODELS, PARAMETERS & BENCHMARK RESULTS
           Model           Total Params Trainable Params                               LoRA Config                                                Training Hyperparameters                                         Eval Decoding HE Pass@1 HE Fix@3 HE Fix@5 HE CondRepair@3 HE CondRepair@5 MBPP Pass@1 MBPP Fix@3 MBPP Fix@5 MBPP CondRepair@3 MBPP CondRepair@5
  Zero-Shot Base 1.346B (1,346,471,936)        0 (0.00%)                        None (Frozen Base)                                   Zero-shot pretrained (no fine-tuning) T1: Greedy | T2..5: temp=1.0, top_p=0.95, max_tok=320    65.24%   70.12%   71.95%          14.04%          19.30%      39.00%     40.80%     43.20%             2.95%             6.89%
    SFT Baseline 1.350B (1,349,617,664)    3.15M (0.23%) r=16, alpha=32, drop=0.05 (q_proj,v_proj)                         LR=2e-5, epochs=1, batch=2x4, seed=42 (APPS 2k) T1: Greedy | T2..5: temp=1.0, top_p=0.95, max_tok=320  

,Model,Total Params,Trainable Params,LoRA Config,Training Hyperparameters,Eval Decoding,HE Pass@1,HE Fix@3,HE Fix@5,HE CondRepair@3,HE CondRepair@5,MBPP Pass@1,MBPP Fix@3,MBPP Fix@5,MBPP CondRepair@3,MBPP CondRepair@5
0,Zero-Shot Base,"1.346B (1,346,471,936)",0 (0.00%),None (Frozen Base),Zero-shot pretrained (no fine-tuning),"T1: Greedy | T2..5: temp=1.0, top_p=0.95, max_tok=320",65.24%,70.12%,71.95%,14.04%,19.30%,39.00%,40.80%,43.20%,2.95%,6.89%
1,SFT Baseline,"1.350B (1,349,617,664)",3.15M (0.23%),"r=16, alpha=32, drop=0.05 (q_proj,v_proj)","LR=2e-5, epochs=1, batch=2x4, seed=42 (APPS 2k)","T1: Greedy | T2..5: temp=1.0, top_p=0.95, max_tok=320",60.98%,63.41%,65.85%,6.25%,12.50%,37.80%,41.00%,43.00%,5.14%,8.36%
2,PPO-Dense Model,"1.350B (1,349,619,713)",3.15M (0.23%),"r=16, alpha=32, drop=0.05 (q_proj,v_proj)","reward=dense, LR=2e-07, steps=100, init_KL=0.3, target_KL=4.0, seed=42","T1: Greedy | T2..5: temp=1.0, top_p=0.95, max_tok=320",62.80%,64.63%,65.85%,4.92%,8.20%,38.00%,41.00%,42.40%,4.84%,7.10%
3,PPO-Binary Model,"1.350B (1,349,619,713)",3.15M (0.23%),"r=16, alpha=32, drop=0.05 (q_proj,v_proj)","reward=binary, LR=2e-07, steps=100, init_KL=0.3, target_KL=4.0, seed=42","T1: Greedy | T2..5: temp=1.0, top_p=0.95, max_tok=320",60.37%,65.24%,66.46%,12.31%,15.38%,38.20%,41.60%,43.20%,5.50%,8.09%
4,DPO Model,"1.350B (1,349,617,664)",3.15M (0.23%),"r=16, alpha=32, drop=0.05 (q_proj,v_proj)","beta=0.1, LR=5e-05, epochs=3, batch=4, pairs=22, seed=42","T1: Greedy | T2..5: temp=1.0, top_p=0.95, max_tok=320",63.41%,66.46%,68.90%,8.33%,15.00%,39.20%,42.20%,44.00%,4.93%,7.89%
